# Cloud Native Satellite Data Demo 1
* Use pystac_client to search the STAC API at Planetary Computer
* Use odc.stac to load the data
* Use hvplot to visualize the data

In [ ]:
import pystac_client
import planetary_computer
from rich.table import Table
import hvplot.xarray

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [ ]:
collections = list(catalog.get_all_collections())
collections.sort(key=lambda c: c.id)
table = Table("ID", "Title", title="Planetary Computer collections")
for collection in collections:
    table.add_row(collection.id, collection.title)
#table

In [ ]:
# Define our area of interest (AOI) - a rough box around Cape Cod gulf of tunisia
bbox = [10.0, 36.6, 10.9, 37.2]

In [ ]:
# Let's search for Sentinel-3 SLSTR WST (Sea Surface Temperature) Level-2 data  NADIA
search = catalog.search(
    collections=["sentinel-3-slstr-wst-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

# Get the results and see what we found
items = search.item_collection()
print(f"Found {len(items)} scenes matching your criteria.")

# Let's inspect the assets of the first item to see the available data bands
if items:
    first_item = items[0]
    print("\nAssets for the first scene:")
    for asset_key, asset in first_item.assets.items():
        print(f"- {asset_key}: {asset.title}")

In [ ]:
import xarray as xr
import fsspec

# Prendre le premier item
item = items[0]

# Signer l'URL
asset = item.assets["l2p"]
signed_asset = planetary_computer.sign(asset)

print(f"URL : {signed_asset.href}")

# Ouvrir avec xarray
ds = xr.open_dataset(fsspec.open(signed_asset.href).open())
print(ds)

In [ ]:
import numpy as np

# Charger seulement lat/lon d'abord (légères)
lat = ds["lat"].values
lon = ds["lon"].values

# Masque spatial : golfe de Tunis
mask = (
    (lat >= 36.6) & (lat <= 37.2) &
    (lon >= 10.0) & (lon <= 10.9)
)

# Indices valides
nj_idx, ni_idx = np.where(mask)

if len(nj_idx) == 0:
    print("❌ Aucun pixel dans la bbox — essaie un autre item")
else:
    print(f"✅ {len(nj_idx)} pixels trouvés dans le golfe de Tunis")

    # Slice les indices min/max pour extraire un bloc rectangulaire
    nj_min, nj_max = nj_idx.min(), nj_idx.max()
    ni_min, ni_max = ni_idx.min(), ni_idx.max()

    # Extraire la SST sur cette zone seulement
    sst_subset = ds["sea_surface_temperature"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max),
        time=0
    ).load()

    lat_subset = ds["lat"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max)
    ).values

    lon_subset = ds["lon"].isel(
        nj=slice(nj_min, nj_max),
        ni=slice(ni_min, ni_max)
    ).values

    # Convertir Kelvin → Celsius
    sst_celsius = sst_subset - 273.15

    print(f"SST min: {float(sst_celsius.min()):.1f}°C")
    print(f"SST max: {float(sst_celsius.max()):.1f}°C")

In [ ]:
import numpy as np
import xarray as xr
import hvplot.pandas
import pandas as pd



# Filtrer les valeurs invalides (< 10°C = terre ou nuage pour la Méditerranée)
sst_vals = sst_celsius.values.flatten()
lat_vals = lat_subset.flatten()
lon_vals = lon_subset.flatten()

# Créer un DataFrame et filtrer
df = pd.DataFrame({
    "lat": lat_vals,
    "lon": lon_vals,
    "sst": sst_vals
})

# Garder uniquement les pixels valides
df = df[(df["sst"] > 10) & (df["sst"] < 40)]

print(f"Pixels valides : {len(df)}")
print(f"SST min: {df['sst'].min():.1f}°C")
print(f"SST max: {df['sst'].max():.1f}°C")
print(f"SST moyenne: {df['sst'].mean():.1f}°C")

# Visualisation hvplot

plot = df.hvplot.points(
    x="lon", y="lat",
    c="sst",
    colormap="RdYlBu_r",
    title="Sea Surface Temperature — Gulf of Tunisia ",
    xlabel="Longitude",
    ylabel="Latitude",
    clabel="SST (°C)",
    size=8,
    geo=True,
    tiles="OSM",
    width=700,
    height=500
)

plot  # ← suffit dans Jupyter, pas besoin de hvplot.show()